In [ ]:
pip install requests beautifulsoup4 pandas
pip install requests
pip install pandas

In [3]:
import requests
import bs4
import pandas as pd

print("All libraries installed successfully!")

All libraries installed successfully!


In [ ]:
# Page 1 → catalog/page-1.html
# Page 2 → catalog/page-2.html
# Page 3 → catalog/page-3.html
...

In [ ]:
# 6. Step 4 – Inspect HTML tags
# Website open karke kisi book par right-click → Inspect kijiye.

# Aapko kuch HTML tags milenge:
https://books.toscrape.com/catalogue/page-2.html


In [ ]:
https://books.toscrape.com/catalogue/page-3.html

In [ ]:
https://books.toscrape.com/catalogue/page-4.html

In [ ]:
https://books.toscrape.com/catalogue/page-5.html

In [ ]:


from urllib.parse import urljoin
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# --------------------------------
# 1. Logging Configuration
# --------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


# --------------------------------
# 2. Website Configuration
# --------------------------------

BASE_URL = "https://books.toscrape.com/"
START_URL = urljoin(BASE_URL, "catalogue/page-1.html")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}


# --------------------------------
# 3. Create Session with Retries
# --------------------------------

session = requests.Session()
session.headers.update(HEADERS)

retry_strategy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry_strategy)

session.mount("https://", adapter)
session.mount("http://", adapter)


# --------------------------------
# 4. Function to Scrape One Page
# --------------------------------

def scrape_page(url):

    try:

        response = session.get(url, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        products = []

        books = soup.select("article.product_pod")

        for book in books:

            title_tag = book.select_one("h3 a")
            price_tag = book.select_one(".price_color")
            rating_tag = book.select_one(".star-rating")
            availability_tag = book.select_one(".availability")

            title = (
                title_tag.get("title", "").strip()
                if title_tag else None
            )

            price = (
                price_tag.get_text(strip=True)
                if price_tag else None
            )

            rating = (
                rating_tag.get("class", [])[-1]
                if rating_tag and len(rating_tag.get("class", [])) > 1
                else None
            )

            availability = (
                availability_tag.get_text(" ", strip=True)
                if availability_tag else None
            )

            product_url = (
                urljoin(url, title_tag.get("href"))
                if title_tag and title_tag.get("href")
                else None
            )

            product = {
                "Product_Name": title,
                "Price": price,
                "Rating": rating,
                "Availability": availability,
                "Product_URL": product_url
            }

            products.append(product)

        logging.info(
            f"Scraped {len(products)} products from {url}"
        )

        next_button = soup.select_one("li.next a")

        next_url = (
            urljoin(url, next_button.get("href"))
            if next_button else None
        )

        return products, next_url

    except requests.exceptions.RequestException as error:

        logging.error(f"Request failed: {error}")
        return [], None

    except Exception as error:

        logging.exception(f"Unexpected error: {error}")
        return [], None


# --------------------------------
# 5. Pagination Loop
# --------------------------------

def scrape_all_pages():

    all_products = []

    current_url = START_URL

    page_number = 1

    max_pages = 50

    while current_url and page_number <= max_pages:

        logging.info(f"Scraping Page {page_number}")

        products, next_url = scrape_page(current_url)

        all_products.extend(products)

        current_url = next_url

        page_number += 1

        time.sleep(1)

    return all_products


# --------------------------------
# 6. Export Data to CSV and JSON
# --------------------------------

def export_data(products):

    if not products:
        logging.warning("No data available to export.")
        return

    df = pd.DataFrame(products)

    df.to_csv(
        "products.csv",
        index=False,
        encoding="utf-8-sig"
    )

    with open(
        "products.json",
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            products,
            file,
            indent=4,
            ensure_ascii=False
        )

    logging.info("CSV file created successfully!")
    logging.info("JSON file created successfully!")


# --------------------------------
# 7. Main Function
# --------------------------------

def main():

    logging.info("Web Scraping Project Started")

    products = scrape_all_pages()

    logging.info(
        f"Total Products Scraped: {len(products)}"
    )

    export_data(products)

    logging.info("Project Completed Successfully!")


if __name__ == "__main__":
    main()


2026-11-03 19:48:47,112 - INFO - Web Scraping Project Started
2026-11-03 19:48:47,114 - INFO - Scraping Page 1
2026-11-03 19:48:49,558 - INFO - Scraped 20 products from https://books.toscrape.com/catalogue/page-1.html
2026-11-03 19:48:50,578 - INFO - Scraping Page 2
2026-11-03 19:48:51,355 - INFO - Scraped 20 products from https://books.toscrape.com/catalogue/page-2.html
2026-11-03 19:48:52,380 - INFO - Scraping Page 3
2026-11-03 19:48:53,709 - INFO - Scraped 20 products from https://books.toscrape.com/catalogue/page-3.html
2026-11-03 19:48:54,721 - INFO - Scraping Page 4
2026-11-03 19:48:55,620 - INFO - Scraped 20 products from https://books.toscrape.com/catalogue/page-4.html
2026-11-03 19:48:56,636 - INFO - Scraping Page 5
2026-11-03 19:48:57,562 - INFO - Scraped 20 products from https://books.toscrape.com/catalogue/page-5.html
2026-11-03 19:48:58,585 - INFO - Scraping Page 6
2026-11-03 19:48:59,478 - INFO - Scraped 20 products from https://books.toscrape.com/catalogue/page-6.html
20

In [5]:
[
    {
        "Product_Name": "A Light in the Attic",
        "Price": "£51.77",
        "Rating": "Three",
        "Availability": "In stock",
        "Product_URL": "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
    }
]

[{'Product_Name': 'A Light in the Attic',
  'Price': '£51.77',
  'Rating': 'Three',
  'Availability': 'In stock',
  'Product_URL': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}]

In [ ]:
pip install -r requirements.txt

In [ ]:
https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html

In [ ]:
#PRODCT DETIALE
Product Description
It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel
Silverstein celebrates its 20th anniversary with this special edition. Silverstein's 
humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and 
read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic.
    This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with 
this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers.
Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. 
    Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?
    And who put you up there,And your cradle, too?Baby, I think someone down here'sGot it in for you. Shel, you never sounded so good. ...more